In [1]:
# ------------------------------------------
# NOTEBOOK DE CONSOLIDAÇÃO DE FEATURE ENGINEERING
# ------------------------------------------

In [2]:
# ------------------------------------------
# TAXA DE CONVERSAO BRL/USD : 4.40
# ------------------------------------------
PTAX = 4.40

In [3]:
# ------------------------------------------
# modulos pip
# ------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [4]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
# ------------------------------------------
# DATABASE: historico dos couriers churneados pela plataforma
#
# SK.CREATED_AT::DATE : data de cadastro do courier na plataforma
# FECHA_ULT : ultima data de acesso do app pelo courier
#
# NOTA: esse database foi recebido com linhas duplicadas
# apos tratamento, o total de linhas caiu de 32 milhões para 152,2 mil
# ------------------------------------------
df_churn = pd.read_csv("/content/drive/Shareddrives/grupo4-rappi-hour/bases-rappi/criacao-contas-churn-sem-duplicados-prov.csv")
df_churn.drop(labels="Unnamed: 0", axis=1, inplace = True)
print(f"Total de linhas: {df_churn.shape[0]}")
df_churn.head(3)

Total de linhas: 152224


,ID,FIRST_NAME,GENDER,CITY,SK.CREATED_AT::DATE,TRANSPORT_MEDIA_TYPE,CARTAO,LEVEL_NAME,FECHA_ULT
0,1286316,Adailton,M,Grande São Paulo,2021-06-07,motorbike,True,bronze,2022-01-16T23:27:35Z
1,1110698,Adriano Floriano Da Silva,M,Recife,2021-02-11,bicycle,True,bronze,2021-07-15T11:16:04Z
2,284886,Bruno,M,Grande São Paulo,2019-07-03,motorbike,False,bronze,2021-07-07T12:33:21Z


In [6]:
# ------------------------------------------
# database contem dados de cidades estrangeiras e nulos
# exemplo : Tuxtla Gutierrez [MX] ; without_city
#
# 80% do churn está concentrado nas seguintes cidades
# ------------------------------------------
df_conc_churn = (df_churn["CITY"].value_counts(normalize=True).cumsum() < 0.8).to_frame()
df_conc_by_city = df_conc_churn[df_conc_churn["CITY"] == True]
df_conc_by_city.index

Index(['Grande São Paulo', 'Rio de Janeiro', 'Belo Horizonte', 'Recife',
       'Curitiba', 'Fortaleza', 'Porto Alegre'],
      dtype='object')

In [7]:
# ------------------------------------------
# database contem couriers que abandonaram a plataforma
# entre junho de 2021 a julho de 2022
# ------------------------------------------
df_churn["FECHA_ULT"].map(lambda x : x[:7]).value_counts().sort_index()

2021-06    11633
2021-07    12387
2021-08    11777
2021-09     9838
2021-10     9760
2021-11     8653
2021-12     8374
2022-01     9337
2022-02     9966
2022-03    13781
2022-04    13602
2022-05    15140
2022-06    16030
2022-07     1946
Name: FECHA_ULT, dtype: int64

In [8]:
# ------------------------------------------
# CONCLUSAO - FEATURE ENGINEERING - DATABASES:
#     . CRIACAO CONTAS CHURN
#
# dataframe dos couriers que churnearam 
# a plataforma e atuam nas cidades de maior volume
# ------------------------------------------
filter = df_conc_by_city.index.to_list()
df_churn_conc_by_city = df_churn[df_churn['CITY'].isin(filter)]
print(f"Total de linhas: {df_churn_conc_by_city.shape[0]}")
df_churn_conc_by_city.head(3)

Total de linhas: 120206


,ID,FIRST_NAME,GENDER,CITY,SK.CREATED_AT::DATE,TRANSPORT_MEDIA_TYPE,CARTAO,LEVEL_NAME,FECHA_ULT
0,1286316,Adailton,M,Grande São Paulo,2021-06-07,motorbike,True,bronze,2022-01-16T23:27:35Z
1,1110698,Adriano Floriano Da Silva,M,Recife,2021-02-11,bicycle,True,bronze,2021-07-15T11:16:04Z
2,284886,Bruno,M,Grande São Paulo,2019-07-03,motorbike,False,bronze,2021-07-07T12:33:21Z


In [9]:
# ------------------------------------------
# DATABASE : informacoes gerais de couriers ativos na plataforma
#
# is_active (bool) : true se ativo ; false se inativo
# ------------------------------------------
df_infos_gerais = pd.read_csv("/content/drive/Shareddrives/grupo4-rappi-hour/bases-rappi/infos-gerais-prov.csv")
print(f"Total de linhas: {df_infos_gerais.shape[0]}")
perc_ativos = round(df_infos_gerais["IS_ACTIVE"].value_counts(normalize=True)[1] * 100, 1)
print(f"% de couriers ativos no database: {perc_ativos}%")
df_infos_gerais.head(3)

Total de linhas: 180178
% de couriers ativos no database: 99.8%


,ID,NOME,SOBRENOME,GENERO,DATA_NASCIMENTO,CIDADE,IS_ACTIVE,TRANSPORTE,AUTO_ACEITE,COUNT_ORDERS_LAST_7D,...,ULTIMO_PEDIDO,COUNT_ORDERS_RESTAURANTES,COUNT_ORDERS_MERCADO,COUNT_ORDERS_FARMACIA,COUNT_ORDERS_EXPRESS,COUNT_ORDERS_ECOMMERCE,COUNT_ORDERS_ANTOJO,FRETE_MEDIO,COOKING_TIME_MEDIO,ITENS_MEDIO
0,1561246,Wilton Jhonne,Da Silva Abreu,M,1988-04-21,Sao Paulo,True,motorbike,True,1,...,2022-08-01T00:00:00Z,1,0,0,0,0,0,62.255500,10.000000,1.0
1,1561210,Dennis Leonardo,Pereira Santos,M,1998-06-28,Grande São Paulo,True,motorbike,True,7,...,2022-08-01T00:00:00Z,6,1,0,0,0,0,43.444714,23.142857,3.0
2,1561205,Thavillo Teles,Balviano De Olivetra,M,1997-09-25,Natal,True,motorbike,True,2,...,2022-08-01T00:00:00Z,1,1,0,0,0,0,42.230500,4.500000,3.5


In [14]:
# ------------------------------------------
# padronizacao de grafia das cidades usadas para
# filtragem dos couriers
# ------------------------------------------
df_infos_gerais.replace(
    {"CIDADE": {
        'Sao Paulo': 'Grande São Paulo',
        'São Paulo': 'Grande São Paulo',
        'SÃO PAULO': 'Grande São Paulo',
        'BELO HORIZINTE': 'Belo Horizonte',
        'RIO DE JANEIRO': 'Rio de Janeiro'
    }}, inplace=True
)

In [17]:
# ------------------------------------------
# filtragem das infos gerais para 
# ------------------------------------------
df_infos_gerais_conc_by_city = df_infos_gerais[df_infos_gerais['CIDADE'].isin(filter)]
df_infos_gerais_conc_by_city["CIDADE"].unique()

array(['Grande São Paulo', 'Recife', 'Fortaleza', 'Belo Horizonte',
       'Rio de Janeiro', 'Porto Alegre', 'Curitiba'], dtype=object)

In [ ]:
# ------------------------------------------
# cruzamento dos couriers inativos de acordo com o database de churn
# e os dados gerais de couriers ativos do database de infos gerais
# 
# RESULTADO ESPERADO : baixa repeticao de IDs entre os databases
# RESULTADO OBSERVADO : alta repeticao de IDs (101.382)
#
# CONCLUSAO : repeticao elevada reflete couriers que abandonam a
# plataforma periodicamente e acabam retornando apos um periodo
# ------------------------------------------
filter = df_churn_conc_by_city["ID"].to_list()
df_infos_gerais[df_infos_gerais["ID"].isin(filter)].shape[0]

101382

In [ ]:
# ------------------------------------------
# EXEMPLO (part 1 de 2):
# id 1544047 - Raquel da Costa Mendes, consta como churn, tendo
# acessado a plataforma pela ultima vez em 03-07-2022
# ------------------------------------------
print(df_churn_conc_by_city[df_churn_conc_by_city["ID"] == 1544047])

            ID FIRST_NAME GENDER            CITY SK.CREATED_AT::DATE  \
44151  1544047     Raquel      F  Belo Horizonte          2022-07-03   

      TRANSPORT_MEDIA_TYPE CARTAO LEVEL_NAME             FECHA_ULT  
44151                  car  False     rookie  2022-07-03T20:29:39Z  


In [ ]:
# ------------------------------------------
# EXEMPLO (part 2 de 2):
# id 1544047 - Raquel da Costa Mendes, consta com status ATIVO
# ------------------------------------------
print(df_infos_gerais[df_infos_gerais["ID"] == 1544047])

           ID    NOME        SOBRENOME GENERO DATA_NASCIMENTO          CIDADE  \
3572  1544047  Raquel  Da Costa Mendes      F      1983-03-01  Belo Horizonte   

      IS_ACTIVE TRANSPORTE  AUTO_ACEITE  COUNT_ORDERS_LAST_7D  ...  \
3572       True        car         True                     0  ...   

             ULTIMO_PEDIDO  COUNT_ORDERS_RESTAURANTES  COUNT_ORDERS_MERCADO  \
3572  2022-07-03T00:00:00Z                          0                     0   

      COUNT_ORDERS_FARMACIA COUNT_ORDERS_EXPRESS COUNT_ORDERS_ECOMMERCE  \
3572                      0                    0                      0   

      COUNT_ORDERS_ANTOJO  FRETE_MEDIO  COOKING_TIME_MEDIO  ITENS_MEDIO  
3572                    0      53.3555                 NaN          2.0  

[1 rows x 25 columns]


In [ ]:
# ------------------------------------------
# CONCLUSAO - FEATURE ENGINEERING - DATABASES:
#     . CRIACAO CONTAS CHURN
#     . INFOS GERAIS 
#
# DECISAO : tierizar os dados de churn (col: IS_ACTIVE)
#     . categoria 1 : couriers que nunca foram churneados
#     . categoria 2 : couriers inconsistentes
#     . categoria 3 : couriers com churn definitivo
#     . base de decisao : database CRIACAO CONTAS CHURN
# ------------------------------------------
df_infos_gerais.head(2)

,ID,NOME,SOBRENOME,GENERO,DATA_NASCIMENTO,CIDADE,IS_ACTIVE,TRANSPORTE,AUTO_ACEITE,COUNT_ORDERS_LAST_7D,...,ULTIMO_PEDIDO,COUNT_ORDERS_RESTAURANTES,COUNT_ORDERS_MERCADO,COUNT_ORDERS_FARMACIA,COUNT_ORDERS_EXPRESS,COUNT_ORDERS_ECOMMERCE,COUNT_ORDERS_ANTOJO,FRETE_MEDIO,COOKING_TIME_MEDIO,ITENS_MEDIO
0,1561246,Wilton Jhonne,Da Silva Abreu,M,1988-04-21,Sao Paulo,True,motorbike,True,1,...,2022-08-01T00:00:00Z,1,0,0,0,0,0,62.255500,10.000000,1.0
1,1561210,Dennis Leonardo,Pereira Santos,M,1998-06-28,Grande São Paulo,True,motorbike,True,7,...,2022-08-01T00:00:00Z,6,1,0,0,0,0,43.444714,23.142857,3.0


In [ ]:
filter = df_churn_conc_by_city["ID"].to_list()
mask = df_infos_gerais["ID"].isin(filter)
column_name = "IS_ACTIVE"
df_infos_gerais.loc[mask, column_name] = False

In [ ]:
df_infos_gerais["IS_ACTIVE"].value_counts(normalize=True)

False    0.564769
True     0.435231
Name: IS_ACTIVE, dtype: float64